# Thực nghiệm mở rộng mô hình Hán cổ (guwen-ner · SikuBERT · kraken GPU)

**Repo:** `family-tree` · nhánh `claude/meeting-project-evaluation-eui68s` · tạo 2026-09-28
**Liên quan:** `docs/planning/chinese_genealogy_model_hunt_plan.md` (§0, §6), `research/model_survey/EVALUATION.md` (mục "Tái lập độc lập 27/9"), `docs/planning/hannom_ner_model_upgrade_proposal.md`.

## Cách dùng
1. Mở trên Google Colab → **Runtime → Change runtime type → GPU** (bắt buộc cho Cell C; A/B chạy được trên CPU).
2. Mỗi cell code **tự chứa**: tự cài dependency, tự tải dữ liệu, tự lưu kết quả vào `/content/results/*.json`. Chạy riêng lẻ cell nào cũng được, không phụ thuộc biến của cell khác.
3. Chạy xong: **File → Download → .ipynb** (giữ output) và push lại vào `research/model_survey/colab/`. Cell cuối nén `/content/results/` để tải JSON kèm theo.

## Nguồn (pin commit / revision để tái lập)
| Thành phần | Nguồn | Pin |
|---|---|---|
| Văn bản phồn thể | [kanripo/KR2a0001](https://github.com/kanripo/KR2a0001) (史記, Kanripo) | `1c19dc6` |
| guwen-ner | [ethanyt/guwen-ner](https://huggingface.co/ethanyt/guwen-ner) · Apache-2.0 | `591273f` |
| GuwenBERT base | [ethanyt/guwenbert-base](https://huggingface.co/ethanyt/guwenbert-base) · Apache-2.0 | `eff0d4a` |
| SikuBERT | [SIKU-BERT/sikubert](https://huggingface.co/SIKU-BERT/sikubert) · license kiểm tra lại trong Cell B | `fc656de` |
| CHAT_models | [colibrisson/CHAT_models](https://github.com/colibrisson/CHAT_models) · CC BY-NC 4.0 (chỉ nghiên cứu) | `9b86ab6` |

## Giới hạn cần nêu khi báo cáo
- **Gold label** (tên người / chức tước) do agent gán tay — cần người rà lại trước khi trích số.
- 史記 gần như chắc chắn nằm trong corpus pretrain của GuwenBERT (殆知阁, đã giản thể hoá) → dữ liệu *in-distribution* về nội dung, **thuận lợi** cho model. Nếu vẫn ra ~0% trên phồn thể thì kết luận càng mạnh; nếu ra cao thì chưa chắc suy rộng sang gia phả Hán-Nôm Việt.
- Đây là 史記 (TQ), không phải gia phả Việt — các đoạn được chọn là đoạn **dạng phả hệ** (X生Y, 父曰…, 其先…, 字…) để gần phong cách gia phả nhất.
- Bản giản thể do **OpenCC t2s** sinh ra (chuyển máy, không phải văn bản giản thể gốc).

In [ ]:
# Cell 0 — Thông tin môi trường (tuỳ chọn, chỉ để ghi lại)
import sys, platform, subprocess
print("python :", sys.version.split()[0])
print("platform:", platform.platform())
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "Không có GPU — Cell C cần Runtime → Change runtime type → GPU")

## Nhóm A — Falsification check cho guwen-ner (phồn thể vs giản thể)

**Câu hỏi.** Kết quả 27/9 (0/4 thực thể trên *một* câu phồn thể tự soạn, 3/4 trên bản giản thể) có khái quát được không, hay chỉ là hiện tượng của một câu (vd. riêng chữ `諱` → `[UNK]`)?

**Biến độc lập:** dạng chữ (phồn gốc Kanripo / giản do OpenCC t2s). **Biến phụ thuộc:** recall tên người (và chức tước, đếm riêng). **Kiểm soát:** cùng đoạn văn, cùng model + revision, cùng `aggregation_strategy="simple"`, cùng tiêu chí khớp. **Đối chứng dương:** `P0_control` = đúng câu của log 27/9 — kỳ vọng tái lập 0/3 tên (phồn) và 3/3 tên (giản).

**Tiêu chí khớp.** *exact* = span dự đoán trùng nguyên văn gold; *lenient* = span chứa gold (vd. `吳太伯` ⊇ `太伯`) hoặc là chuỗi con ≥2 ký tự của gold. Báo cáo riêng tập tên ≥2 ký tự vì tên 1 chữ (`羽`, `季`, `平`…) khó hơn hẳn.

**Tiêu chí quyết định (đặt trước khi chạy, tính trên P1–P9, recall lenient):**
- **Ủng hộ** "guwen-ner gần như vô dụng trên phồn thể": phồn ≤ 10% **và** giản ≥ 50%.
- **Bác bỏ**: phồn ≥ 50% → kết quả 0/4 hôm 27/9 là hiện tượng riêng của câu đó.
- Ngoài hai vùng trên: **chưa kết luận**, ghi nguyên số.
- **Bổ sung sau smoke test local 28/9 (post-hoc, KHÔNG thuộc tiêu chí đặt trước):** (i) điều kiện **bỏ dấu câu** (2×2: dạng chữ × dấu câu) — câu đối chứng P0 không có dấu câu còn đoạn 史記 có, và gia phả OCR thật thường không có dấu câu; (ii) đếm **span chứa ký tự `[UNK]` ngoài gold** (span dự đoán có ký tự `[UNK]` và không trùng tên/chức gold nào). Chỉ là chỉ báo nhiễu thô: vì chưa gán nhãn địa danh, cột này gồm cả địa danh đúng (`魯`, `隴西`…) lẫn nhiễu thật (`時`, `為將`, `讀`, `書`…) → **phải rà tay** cột danh sách span; không phải precision.
- **Cơ chế:** nếu các đoạn phồn thể thất bại đồng thời có tỷ lệ `[UNK]` cao → nguyên nhân là độ phủ vocab (vocab GuwenBERT là giản thể). Nếu `[UNK]` thấp mà vẫn thất bại → cơ chế khác, cần điều tra thêm.

In [ ]:
# === Cell A — guwen-ner trên đoạn 史記 thật: phồn thể vs giản thể ===
import subprocess, sys
TRANSFORMERS_PIN = "4.46.3"   # khớp log 26–27/9; đặt "" để dùng bản Colab có sẵn nếu pip lỗi
if "transformers" in sys.modules:
    print("⚠️ transformers đã được import trong runtime này — nếu khác version pin, hãy Runtime → Restart session rồi chạy lại cell.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"transformers=={TRANSFORMERS_PIN}" if TRANSFORMERS_PIN else "transformers",
                "opencc-python-reimplemented"], check=True)

import json, re, math, datetime, platform, pathlib, urllib.request
from importlib.metadata import version

# ---- Bộ test (lặp nguyên văn ở Cell A và Cell B — cố ý, để mỗi cell tự chứa) ----
KANRIPO_REPO = "kanripo/KR2a0001"   # 史記 — Kanripo, base edition "tls", chữ phồn thể
KANRIPO_SHA = "1c19dc6fa970b1c530fced9e8e3697d19163c26c"
# per   = tên người; title = chức/tước (đếm riêng).
# Gold do agent gán tay 2026-09-28 — CẦN NGƯỜI RÀ LẠI trước khi trích số vào luận văn.
TESTSET = [
    {"id": "P0_control", "chapter": "câu tự soạn 26–27/9 (đối chứng dương, KHÔNG phải nguồn thật)", "file": None,
     "text": "始祖諱文成官至知府娶阮氏生子一人諱德重", "per": ["文成", "阮氏", "德重"], "title": ["知府"]},
    {"id": "P1_kongzi", "chapter": "孔子世家", "file": "KR2a0001_400.txt",
     "text": "孔子生魯昌平鄉陬邑。其先宋人也，曰孔防叔。防叔生伯夏，伯夏生叔梁紇。紇與顏氏女野合而生孔子",
     "per": ["孔子", "孔防叔", "伯夏", "叔梁紇", "顏氏"], "title": []},
    {"id": "P2_taibo", "chapter": "吳太伯世家", "file": "KR2a0001_400.txt",
     "text": "吳太伯，太伯弟仲雍，皆周太王之子，而王季歷之兄也。季歷賢，而有聖子昌",
     "per": ["太伯", "仲雍", "太王", "季歷", "昌"], "title": []},
    {"id": "P3_xiangyu", "chapter": "項羽本紀", "file": "KR2a0001_100.txt",
     "text": "項籍者，下相人也，字羽。初起時，年二十四。其季父項梁，梁父即楚將項燕，為秦將王翦所戮者也。",
     "per": ["項籍", "羽", "項梁", "項燕", "王翦"], "title": []},
    {"id": "P4_gaozu", "chapter": "高祖本紀", "file": "KR2a0001_100.txt",
     "text": "高祖，沛豐邑中陽里人，姓劉氏，字季。父曰太公，母曰劉媼。",
     "per": ["高祖", "季", "太公", "劉媼"], "title": []},
    {"id": "P5_laozi", "chapter": "老子韓非列傳", "file": "KR2a0001_500.txt",
     "text": "老子者，楚苦縣厲鄉曲仁里人也，姓李氏，名耳，字聃，周守藏室之史也。",
     "per": ["老子", "耳", "聃"], "title": ["守藏室之史"]},
    {"id": "P6_mengchang", "chapter": "孟嘗君列傳", "file": "KR2a0001_500.txt",
     "text": "孟嘗君名文，姓田氏。文之父曰靖郭君田嬰，田嬰者，齊威王少子而齊宣王庶弟也。",
     "per": ["孟嘗君", "文", "田嬰", "齊威王", "齊宣王"], "title": ["靖郭君"]},
    {"id": "P7_zhangliang", "chapter": "留侯世家", "file": "KR2a0001_400.txt",
     "text": "留侯張良者，其先韓人也。大父開地，相韓昭侯、宣惠王、襄哀王。父平，相釐王、悼惠王。",
     "per": ["張良", "開地", "韓昭侯", "宣惠王", "襄哀王", "平", "釐王", "悼惠王"], "title": ["留侯"]},
    {"id": "P8_liguang", "chapter": "李將軍列傳", "file": "KR2a0001_500.txt",
     "text": "李將軍廣者，隴西成紀人也。其先曰李信，秦時為將，逐得燕太子丹者也。",
     "per": ["廣", "李信", "太子丹"], "title": ["將軍"]},
    {"id": "P9_chenping", "chapter": "陳丞相世家", "file": "KR2a0001_400.txt",
     "text": "陳丞相平者，陽武戶牖鄉人也。少時家貧，好讀書，有田三十畝，獨與兄伯居。",
     "per": ["平", "伯"], "title": ["丞相"]},
]


def load_kanripo(fname):
    url = f"https://raw.githubusercontent.com/{KANRIPO_REPO}/{KANRIPO_SHA}/{fname}"
    raw = urllib.request.urlopen(url, timeout=60).read().decode("utf-8")
    body = "\n".join(l for l in raw.splitlines() if not l.startswith(("#", "*")))
    body = re.sub(r"<pb:[^>]*>", "", body)
    return re.sub(r"[¶\s\u3000]", "", body)


PUNCT = re.compile(r"[，。、；：「」『』？！（）《》〈〉·]")


def build_testset():
    from opencc import OpenCC
    cc = OpenCC("t2s")
    cache, items = {}, []
    for src in TESTSET:
        it = dict(src)
        for g in it["per"] + it["title"]:
            assert g in it["text"], f"{it['id']}: gold {g!r} không có trong text"
        if it["file"]:
            if it["file"] not in cache:
                cache[it["file"]] = load_kanripo(it["file"])
            assert it["text"] in cache[it["file"]], (
                f"{it['id']}: đoạn trích KHÔNG có nguyên văn trong {it['file']}@{KANRIPO_SHA[:7]}")
            it["source_url"] = f"https://github.com/{KANRIPO_REPO}/blob/{KANRIPO_SHA}/{it['file']}"
        simp = cc.convert(it["text"])
        assert len(simp) == len(it["text"]), f"{it['id']}: OpenCC t2s đổi độ dài chuỗi"
        it["trad"] = {"text": it["text"], "per": it["per"], "title": it["title"]}
        it["simp"] = {"text": simp, "per": [cc.convert(g) for g in it["per"]],
                      "title": [cc.convert(g) for g in it["title"]]}
        for key in ("trad", "simp"):
            d = it[key]
            it[key + "_nopunct"] = {"text": PUNCT.sub("", d["text"]), "per": d["per"], "title": d["title"]}
        items.append(it)
    return items


import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

MODEL_ID, MODEL_REV = "ethanyt/guwen-ner", "591273ff61bb5eb453f47c300f577058d6d2ee15"
items = build_testset()
p0 = next(it for it in items if it["id"] == "P0_control")
print("P0 giản thể (OpenCC):", p0["simp"]["text"],
      "| khớp câu giản thể log 27/9:", p0["simp"]["text"] == "始祖讳文成官至知府娶阮氏生子一人讳德重")

tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV)
mdl = AutoModelForTokenClassification.from_pretrained(MODEL_ID, revision=MODEL_REV)
ner = pipeline("token-classification", model=mdl, tokenizer=tok, aggregation_strategy="simple",
               device=0 if torch.cuda.is_available() else -1)
print("labels:", sorted(set(mdl.config.id2label.values())))


def run_ner(text):
    out = []
    for e in ner(text):
        s, t = e.get("start"), e.get("end")
        surf = text[s:t] if s is not None and t is not None else e["word"].replace(" ", "")
        out.append({"surface": surf, "label": e["entity_group"], "score": round(float(e["score"]), 4)})
    return out


def unk_chars(text):
    return [ch for ch in text if tok.unk_token_id in tok(ch, add_special_tokens=False)["input_ids"]]


def hits(gold, surfaces):
    exact = [g for g in gold if g in surfaces]
    lenient = [g for g in gold if any(g == p or g in p or (len(p) >= 2 and p in g) for p in surfaces)]
    return exact, lenient


SCRIPTS = ("trad", "simp", "trad_nopunct", "simp_nopunct")
LABEL = {"trad": "phồn", "simp": "giản", "trad_nopunct": "phồn, bỏ dấu", "simp_nopunct": "giản, bỏ dấu"}
rows = []
for it in items:
    for script in SCRIPTS:
        d = it[script]
        preds = run_ner(d["text"])
        surf = [p["surface"] for p in preds]
        pe, pl = hits(d["per"], surf)
        multi = [g for g in d["per"] if len(g) >= 2]
        _, ml = hits(multi, surf)
        _, tl = hits(d["title"], surf)
        unk = unk_chars(d["text"])
        gold_all = d["per"] + d["title"]
        spurious_unk = [p for p in surf if any(ch in unk for ch in p)
                        and not any(g in p or p in g for g in gold_all)]
        rows.append({
            "id": it["id"], "script": script, "text": d["text"],
            "n_chars": len(d["text"]), "unk_n": len(unk), "unk_chars": "".join(unk),
            "n_per": len(d["per"]), "per_exact": len(pe), "per_lenient": len(pl),
            "n_per_multi": len(multi), "per_multi_lenient": len(ml),
            "n_title": len(d["title"]), "title_lenient": len(tl),
            "n_pred": len(preds), "spurious_unk_spans": spurious_unk, "n_spurious_unk": len(spurious_unk),
            "missed_per": [g for g in d["per"] if g not in pl],
            "preds": preds,
        })

real = [r for r in rows if r["id"] != "P0_control"]


def micro(script, hit, n):
    rs = [r for r in real if r["script"] == script]
    den = sum(r[n] for r in rs)
    return (sum(r[hit] for r in rs) / den if den else float("nan")), sum(r[hit] for r in rs), den


summary = {}
for script in SCRIPTS:
    summary[script] = {
        "per_exact": micro(script, "per_exact", "n_per"),
        "per_lenient": micro(script, "per_lenient", "n_per"),
        "per_multi_lenient": micro(script, "per_multi_lenient", "n_per_multi"),
        "title_lenient": micro(script, "title_lenient", "n_title"),
        "unk_rate": micro(script, "unk_n", "n_chars"),
        "spurious_unk_per_pred": micro(script, "n_spurious_unk", "n_pred"),
    }

per_passage = {}
for r in (r for r in real if r["script"] in ("trad", "simp")):
    per_passage.setdefault(r["id"], {})[r["script"]] = r["per_lenient"] / r["n_per"]
wins = sum(v["simp"] > v["trad"] for v in per_passage.values())
losses = sum(v["simp"] < v["trad"] for v in per_passage.values())
ties = len(per_passage) - wins - losses
n = wins + losses
sign_p = min(1.0, 2 * sum(math.comb(n, k) for k in range(max(wins, losses), n + 1)) / 2 ** n) if n else float("nan")

pct = lambda a, b: f"{a}/{b} ({a / b:.0%})" if b else "—"
row = lambda i, sc: next(r for r in rows if r["id"] == i and r["script"] == sc)
print("\n### Bảng theo đoạn (recall tên người, lenient)\n")
print("| id | " + " | ".join(LABEL[sc] for sc in SCRIPTS) + " | [UNK] (phồn) | span chứa [UNK] ngoài gold (phồn) |")
print("|---" * (len(SCRIPTS) + 3) + "|")
for it in items:
    cells = [pct(row(it["id"], sc)["per_lenient"], row(it["id"], sc)["n_per"]) for sc in SCRIPTS]
    t = row(it["id"], "trad")
    print(f"| {it['id']} | " + " | ".join(cells) + f" | {t['unk_n']}/{t['n_chars']} {t['unk_chars'] or ''} | "
          f"{' '.join(t['spurious_unk_spans']) or '—'} |")

print("\n### Tổng hợp P1–P9 (micro)\n")
print("| chỉ số | " + " | ".join(LABEL[sc] for sc in SCRIPTS) + " |")
print("|---" * (len(SCRIPTS) + 1) + "|")
for k, label in [("per_exact", "tên người — exact"), ("per_lenient", "tên người — lenient"),
                 ("per_multi_lenient", "tên ≥2 chữ — lenient"), ("title_lenient", "chức tước — lenient"),
                 ("unk_rate", "tỷ lệ ký tự [UNK]"), ("spurious_unk_per_pred", "span chứa [UNK] ngoài gold / tổng span")]:
    print(f"| {label} | " + " | ".join(f"{summary[sc][k][1]}/{summary[sc][k][2]} ({summary[sc][k][0]:.1%})" for sc in SCRIPTS) + " |")
print(f"\nSign test theo đoạn (giản > phồn): thắng {wins}, thua {losses}, hoà {ties}; p hai phía = {sign_p:.4f} (n nhỏ — chỉ tham khảo)")

t0, s0 = row("P0_control", "trad"), row("P0_control", "simp")
print(f"\nĐối chứng dương P0: phồn {t0['per_lenient']}/3, giản {s0['per_lenient']}/3 "
      f"→ {'TÁI LẬP log 27/9' if (t0['per_lenient'], s0['per_lenient']) == (0, 3) else 'KHÁC log 27/9 — kiểm tra môi trường trước khi đọc kết quả P1–P9'}")

def decide(tr, sr):
    if tr >= 0.5:
        return "BÁC BỎ 'vô dụng trên phồn thể' (phồn ≥ 50%)"
    if tr <= 0.10 and sr >= 0.5:
        return "ỦNG HỘ 'gần như vô dụng trên phồn thể'"
    return "CHƯA KẾT LUẬN — báo cáo nguyên số"


verdict = decide(summary["trad"]["per_lenient"][0], summary["simp"]["per_lenient"][0])
verdict_nopunct = decide(summary["trad_nopunct"]["per_lenient"][0], summary["simp_nopunct"]["per_lenient"][0])
print("Kết luận theo tiêu chí đặt trước (có dấu câu):", verdict)
print("Cùng tiêu chí, điều kiện bỏ dấu câu (post-hoc):", verdict_nopunct)
fail = [r for r in real if r["script"] == "trad" and r["per_lenient"] == 0]
ok = [r for r in real if r["script"] == "trad" and r["per_lenient"] > 0]
mean = lambda xs: sum(xs) / len(xs) if xs else float("nan")
print(f"Cơ chế: tỷ lệ [UNK] trung bình ở đoạn phồn thể thất bại = {mean([r['unk_n'] / r['n_chars'] for r in fail]):.1%} "
      f"(n={len(fail)}), ở đoạn có bắt được ≥1 tên = {mean([r['unk_n'] / r['n_chars'] for r in ok]):.1%} (n={len(ok)})")

out = pathlib.Path("/content/results"); out.mkdir(parents=True, exist_ok=True)
json.dump({
    "experiment": "A_guwen_ner_trad_vs_simp",
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "env": {"python": platform.python_version(), "torch": torch.__version__, "transformers": version("transformers"),
            "opencc": version("opencc-python-reimplemented"),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    "model": {"id": MODEL_ID, "revision": MODEL_REV},
    "source": {"repo": KANRIPO_REPO, "sha": KANRIPO_SHA},
    "criteria": {"support": "trad<=10% and simp>=50%", "reject": "trad>=50%", "metric": "micro lenient PER recall P1-P9"},
    "summary": summary, "sign_test": {"wins": wins, "losses": losses, "ties": ties, "p_two_sided": sign_p},
    "verdict": verdict, "verdict_nopunct_posthoc": verdict_nopunct, "rows": rows,
}, open(out / "A_guwen_ner.json", "w"), ensure_ascii=False, indent=1)
print("\nĐã lưu /content/results/A_guwen_ner.json")

## Nhóm B — SikuBERT (lần đầu thử) so với GuwenBERT

**Bước 1 — kiểm tra license (bắt buộc theo §0.3 của plan).** Kiểm tra trước khi viết notebook (2026-09-28): model card HF `SIKU-BERT/sikubert` khai `license: apache-2.0` (tag `license:apache-2.0`, revision `fc656de`); repo GitHub [hsc748NLP/SikuBERT-…](https://github.com/hsc748NLP/SikuBERT-for-digital-humanities-and-classical-Chinese-information-processing) cũng là Apache-2.0. Cell dưới **kiểm tra lại trực tiếp** qua HF API và **dừng, không tải weight** nếu license không thuộc danh sách cho phép.

**Bước 2 — phép thử.** SikuBERT chỉ là model pretrain (fill-mask), **không có head NER** → *không thể* so recall NER trực tiếp với guwen-ner nếu không fine-tune (ngoài phạm vi). Thay vào đó đo **độ phủ vocab và khả năng hiểu ngữ cảnh** trên cùng bộ đoạn của Cell A, so với `ethanyt/guwenbert-base` (model nền của guwen-ner):
- Che (mask) từng ký tự Hán, tính top-1 / top-5 đúng. Tách **ký tự thuộc tên người/chức tước** (`ent`) và **ký tự còn lại** (`ctrl`, mốc nền năng lực ngôn ngữ).
- Ký tự bị tokenizer biến thành `[UNK]` tính là trượt (và đếm riêng).

**Dự đoán:** SikuBERT học trên 四库全书 (phồn thể) → khoảng cách phồn/giản nhỏ; GuwenBERT (corpus đã giản thể hoá) → phồn thể kém hơn rõ. Nếu đúng: SikuBERT là nền hợp lý hơn để fine-tune NER cho gia phả Hán-Nôm (vốn là phồn thể). Nếu GuwenBERT không tụt trên phồn thể ở phép thử này → giả thuyết "thiếu vocab phồn thể" không giải thích được thất bại của guwen-ner.

In [ ]:
# === Cell B — license gate + fill-mask: SikuBERT vs GuwenBERT, phồn vs giản ===
import subprocess, sys
TRANSFORMERS_PIN = "4.46.3"
if "transformers" in sys.modules:
    print("⚠️ transformers đã được import trong runtime này — nếu khác version pin, hãy Runtime → Restart session rồi chạy lại cell.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"transformers=={TRANSFORMERS_PIN}" if TRANSFORMERS_PIN else "transformers",
                "opencc-python-reimplemented"], check=True)

import json, re, datetime, platform, pathlib, urllib.request
from importlib.metadata import version

MODELS = {
    "SIKU-BERT/sikubert": "fc656de2d6bde33919102dd3abe31c843f42226a",
    "ethanyt/guwenbert-base": "eff0d4a5196d7bf7b8be746c5c6437e89d8b9061",
}
ALLOWED = {"apache-2.0", "mit", "bsd", "bsd-2-clause", "bsd-3-clause"}
license_check = {}
for mid in MODELS:
    info = json.load(urllib.request.urlopen(f"https://huggingface.co/api/models/{mid}", timeout=60))
    card = (info.get("cardData") or {}).get("license")
    tags = [t.split(":", 1)[1] for t in info.get("tags", []) if t.startswith("license:")]
    lic = {x.lower() for x in ([card] if card else []) + tags}
    license_check[mid] = {"card": card, "tags": tags, "ok": bool(lic) and lic <= ALLOWED, "head_sha": info.get("sha")}
    print(f"{mid}: card={card!r} tags={tags} head_sha={info.get('sha')} → {'OK' if license_check[mid]['ok'] else 'KHÔNG HỢP LỆ'}")
bad = [m for m, v in license_check.items() if not v["ok"]]
if bad:
    raise SystemExit(f"Dừng: license không thuộc {sorted(ALLOWED)} hoặc không khai báo: {bad}. Không tải weight.")

# ---- Bộ test (lặp nguyên văn ở Cell A và Cell B — cố ý, để mỗi cell tự chứa) ----
KANRIPO_REPO = "kanripo/KR2a0001"   # 史記 — Kanripo, base edition "tls", chữ phồn thể
KANRIPO_SHA = "1c19dc6fa970b1c530fced9e8e3697d19163c26c"
# per   = tên người; title = chức/tước (đếm riêng).
# Gold do agent gán tay 2026-09-28 — CẦN NGƯỜI RÀ LẠI trước khi trích số vào luận văn.
TESTSET = [
    {"id": "P0_control", "chapter": "câu tự soạn 26–27/9 (đối chứng dương, KHÔNG phải nguồn thật)", "file": None,
     "text": "始祖諱文成官至知府娶阮氏生子一人諱德重", "per": ["文成", "阮氏", "德重"], "title": ["知府"]},
    {"id": "P1_kongzi", "chapter": "孔子世家", "file": "KR2a0001_400.txt",
     "text": "孔子生魯昌平鄉陬邑。其先宋人也，曰孔防叔。防叔生伯夏，伯夏生叔梁紇。紇與顏氏女野合而生孔子",
     "per": ["孔子", "孔防叔", "伯夏", "叔梁紇", "顏氏"], "title": []},
    {"id": "P2_taibo", "chapter": "吳太伯世家", "file": "KR2a0001_400.txt",
     "text": "吳太伯，太伯弟仲雍，皆周太王之子，而王季歷之兄也。季歷賢，而有聖子昌",
     "per": ["太伯", "仲雍", "太王", "季歷", "昌"], "title": []},
    {"id": "P3_xiangyu", "chapter": "項羽本紀", "file": "KR2a0001_100.txt",
     "text": "項籍者，下相人也，字羽。初起時，年二十四。其季父項梁，梁父即楚將項燕，為秦將王翦所戮者也。",
     "per": ["項籍", "羽", "項梁", "項燕", "王翦"], "title": []},
    {"id": "P4_gaozu", "chapter": "高祖本紀", "file": "KR2a0001_100.txt",
     "text": "高祖，沛豐邑中陽里人，姓劉氏，字季。父曰太公，母曰劉媼。",
     "per": ["高祖", "季", "太公", "劉媼"], "title": []},
    {"id": "P5_laozi", "chapter": "老子韓非列傳", "file": "KR2a0001_500.txt",
     "text": "老子者，楚苦縣厲鄉曲仁里人也，姓李氏，名耳，字聃，周守藏室之史也。",
     "per": ["老子", "耳", "聃"], "title": ["守藏室之史"]},
    {"id": "P6_mengchang", "chapter": "孟嘗君列傳", "file": "KR2a0001_500.txt",
     "text": "孟嘗君名文，姓田氏。文之父曰靖郭君田嬰，田嬰者，齊威王少子而齊宣王庶弟也。",
     "per": ["孟嘗君", "文", "田嬰", "齊威王", "齊宣王"], "title": ["靖郭君"]},
    {"id": "P7_zhangliang", "chapter": "留侯世家", "file": "KR2a0001_400.txt",
     "text": "留侯張良者，其先韓人也。大父開地，相韓昭侯、宣惠王、襄哀王。父平，相釐王、悼惠王。",
     "per": ["張良", "開地", "韓昭侯", "宣惠王", "襄哀王", "平", "釐王", "悼惠王"], "title": ["留侯"]},
    {"id": "P8_liguang", "chapter": "李將軍列傳", "file": "KR2a0001_500.txt",
     "text": "李將軍廣者，隴西成紀人也。其先曰李信，秦時為將，逐得燕太子丹者也。",
     "per": ["廣", "李信", "太子丹"], "title": ["將軍"]},
    {"id": "P9_chenping", "chapter": "陳丞相世家", "file": "KR2a0001_400.txt",
     "text": "陳丞相平者，陽武戶牖鄉人也。少時家貧，好讀書，有田三十畝，獨與兄伯居。",
     "per": ["平", "伯"], "title": ["丞相"]},
]


def load_kanripo(fname):
    url = f"https://raw.githubusercontent.com/{KANRIPO_REPO}/{KANRIPO_SHA}/{fname}"
    raw = urllib.request.urlopen(url, timeout=60).read().decode("utf-8")
    body = "\n".join(l for l in raw.splitlines() if not l.startswith(("#", "*")))
    body = re.sub(r"<pb:[^>]*>", "", body)
    return re.sub(r"[¶\s\u3000]", "", body)


PUNCT = re.compile(r"[，。、；：「」『』？！（）《》〈〉·]")


def build_testset():
    from opencc import OpenCC
    cc = OpenCC("t2s")
    cache, items = {}, []
    for src in TESTSET:
        it = dict(src)
        for g in it["per"] + it["title"]:
            assert g in it["text"], f"{it['id']}: gold {g!r} không có trong text"
        if it["file"]:
            if it["file"] not in cache:
                cache[it["file"]] = load_kanripo(it["file"])
            assert it["text"] in cache[it["file"]], (
                f"{it['id']}: đoạn trích KHÔNG có nguyên văn trong {it['file']}@{KANRIPO_SHA[:7]}")
            it["source_url"] = f"https://github.com/{KANRIPO_REPO}/blob/{KANRIPO_SHA}/{it['file']}"
        simp = cc.convert(it["text"])
        assert len(simp) == len(it["text"]), f"{it['id']}: OpenCC t2s đổi độ dài chuỗi"
        it["trad"] = {"text": it["text"], "per": it["per"], "title": it["title"]}
        it["simp"] = {"text": simp, "per": [cc.convert(g) for g in it["per"]],
                      "title": [cc.convert(g) for g in it["title"]]}
        for key in ("trad", "simp"):
            d = it[key]
            it[key + "_nopunct"] = {"text": PUNCT.sub("", d["text"]), "per": d["per"], "title": d["title"]}
        items.append(it)
    return items


import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

device = "cuda" if torch.cuda.is_available() else "cpu"
items = build_testset()
real = [it for it in items if it["id"] != "P0_control"]
CJK = re.compile(r"[\u3400-\u9fff\uf900-\ufaff\U00020000-\U0003134f]")


def entity_positions(text, golds):
    pos = set()
    for g in golds:
        for m in re.finditer(re.escape(g), text):
            pos.update(range(m.start(), m.end()))
    return pos


def probe(tok, mlm, text, golds):
    enc = tok(text, return_offsets_mapping=True, return_tensors="pt")
    ids = enc["input_ids"][0]
    char2tok = {s: i for i, (s, e) in enumerate(enc["offset_mapping"][0].tolist()) if e - s == 1}
    ent = entity_positions(text, golds)
    stats = {k: {"n": 0, "top1": 0, "top5": 0, "unk_target": 0, "no_token": 0} for k in ("ent", "ctrl")}
    batch, meta = [], []
    for ci, ch in enumerate(text):
        if not CJK.match(ch):
            continue
        kind = "ent" if ci in ent else "ctrl"
        stats[kind]["n"] += 1
        ti = char2tok.get(ci)
        if ti is None:
            stats[kind]["no_token"] += 1
            continue
        if ids[ti].item() == tok.unk_token_id:
            stats[kind]["unk_target"] += 1
            continue
        x = ids.clone()
        x[ti] = tok.mask_token_id
        batch.append(x)
        meta.append((ci, ti, kind))
    for s in range(0, len(batch), 64):
        X = torch.stack(batch[s:s + 64]).to(device)
        with torch.no_grad():
            logits = mlm(input_ids=X, attention_mask=torch.ones_like(X)).logits
        for row, (ci, ti, kind) in enumerate(meta[s:s + 64]):
            top = tok.convert_ids_to_tokens(logits[row, ti].topk(5).indices.tolist())
            stats[kind]["top1"] += int(top[0] == text[ci])
            stats[kind]["top5"] += int(text[ci] in top)
    return stats


results, per_passage = {}, []
for mid, rev in MODELS.items():
    tok = AutoTokenizer.from_pretrained(mid, revision=rev)
    mlm = AutoModelForMaskedLM.from_pretrained(mid, revision=rev).to(device).eval()
    assert tok.is_fast, f"{mid}: cần fast tokenizer để lấy offset"
    for script in ("trad", "simp"):
        agg = {k: {"n": 0, "top1": 0, "top5": 0, "unk_target": 0, "no_token": 0} for k in ("ent", "ctrl")}
        for it in real:
            d = it[script]
            st = probe(tok, mlm, d["text"], d["per"] + d["title"])
            per_passage.append({"model": mid, "script": script, "id": it["id"], **{f"{k}_{m}": st[k][m] for k in st for m in st[k]}})
            for k in agg:
                for m in agg[k]:
                    agg[k][m] += st[k][m]
        results[(mid, script)] = agg
    del mlm
    if device == "cuda":
        torch.cuda.empty_cache()

pct = lambda a, b: f"{a}/{b} ({a / b:.1%})" if b else "—"
print("\n### Fill-mask P1–P9 (micro)\n")
print("| model | chữ | tên/chức top-1 | tên/chức top-5 | nền top-1 | nền top-5 | ký tự → [UNK] |")
print("|---|---|---|---|---|---|---|")
for (mid, script), a in results.items():
    unk = a["ent"]["unk_target"] + a["ctrl"]["unk_target"]
    tot = a["ent"]["n"] + a["ctrl"]["n"]
    print(f"| {mid} | {script} | {pct(a['ent']['top1'], a['ent']['n'])} | {pct(a['ent']['top5'], a['ent']['n'])} | "
          f"{pct(a['ctrl']['top1'], a['ctrl']['n'])} | {pct(a['ctrl']['top5'], a['ctrl']['n'])} | {pct(unk, tot)} |")

print("\nKhoảng cách giản − phồn (nền top-1, điểm %):")
gaps = {}
for mid in MODELS:
    r = lambda s: results[(mid, s)]["ctrl"]["top1"] / max(1, results[(mid, s)]["ctrl"]["n"])
    gaps[mid] = r("simp") - r("trad")
    print(f"  {mid}: {gaps[mid] * 100:+.1f}")
print("Lưu ý: đây là phép thử fill-mask, KHÔNG phải recall NER — không đặt cạnh số của Cell A như cùng thang đo.")

out = pathlib.Path("/content/results"); out.mkdir(parents=True, exist_ok=True)
json.dump({
    "experiment": "B_sikubert_vs_guwenbert_fillmask",
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "env": {"python": platform.python_version(), "torch": torch.__version__, "transformers": version("transformers"),
            "opencc": version("opencc-python-reimplemented"),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    "license_check": license_check, "models": MODELS, "source": {"repo": KANRIPO_REPO, "sha": KANRIPO_SHA},
    "results": {f"{m}|{s}": v for (m, s), v in results.items()}, "gap_simp_minus_trad_ctrl_top1": gaps,
    "per_passage": per_passage,
}, open(out / "B_sikubert_fillmask.json", "w"), ensure_ascii=False, indent=1)
print("\nĐã lưu /content/results/B_sikubert_fillmask.json")

## Nhóm C — Benchmark GPU cho kraken + CHAT_models

> ⚠️ Yêu cầu gốc cho nhóm C bị cắt sau câu "Trên Colab GPU (bật Runtime → GPU trước khi chạy):". Cell dưới đo theo cách hợp lý nhất đọc được từ ngữ cảnh — **chỉnh cấu hình ở đầu cell** nếu yêu cầu đầy đủ khác.

**Mốc so sánh và vấn đề confound.** Số 459.5 giây/trang (segmentation, CPU, sandbox cloud) trong `EVALUATION.md` được đo với **kraken 7.1.1** — cấu hình đã chứng minh là cho output vô nghĩa. Lần chạy đúng (kraken 4.3.13, 27/9, Mac CPU, `torch.set_num_threads(1)`) chỉ có tổng thời gian **81 phút 18 giây cho 2 ảnh demo**, không tách seg/rec. So GPU với 459.5 giây sẽ lẫn cả phiên bản kraken lẫn phần cứng. Vì vậy cell này đo **cả CPU và GPU trên cùng máy Colab, cùng kraken 4.3.13, cùng ảnh**, tách thời gian segmentation / recognition.

**Môi trường.** kraken 4.3.13 cần Python ≤ 3.11 (log 27/9: Python 3.11.9, torch 2.0.1). Colab mặc định dùng Python mới hơn → cell tạo venv Python 3.11 riêng bằng `uv`, chạy benchmark qua subprocess. Không ảnh hưởng kernel chính.

**Dự đoán (kiểm từ source kraken 4.3.13, `kraken/blla.py`):** `blla.segment` chỉ đưa **forward mạng nơ-ron** lên GPU (`model.to(device)`, dòng 85/115); bước hậu xử lý (vector hoá baseline, dựng polygon) vẫn chạy CPU → tốc độ segmentation tăng có thể **thấp hơn nhiều** so với kỳ vọng "GPU nhanh gấp chục lần". Nếu seg GPU vẫn ~ngang CPU thì nút thắt là hậu xử lý, không phải model.

**Đối chứng dương:** output GPU phải chứa các chuỗi đã thấy trong log 27/9 (`巫山高`, `謝眺`, `恭惟某官`) → xác nhận tái lập H2 trên máy khác. So sánh text CPU vs GPU (difflib) để chắc GPU không đổi kết quả.

**License:** CHAT_models là CC BY-NC 4.0 — chỉ dùng cho nghiên cứu, không đưa vào sản phẩm.

In [ ]:
# === Cell C — kraken 4.3.13 + CHAT_models: CPU vs GPU trên cùng máy Colab ===
RUN_GPU = True
RUN_CPU = True
CPU_THREADS = 1          # = torch.set_num_threads(1) trong demo gốc (cấu hình log 27/9)
CPU_MAX_IMAGES = 1       # CPU rất chậm (27/9: 81 phút cho 2 ảnh trên Mac) — tăng lên 2 nếu có thời gian
CPU_TIMEOUT_S = 2 * 3600
GPU_WARMUP = 1           # số lần segment "làm nóng" CUDA trước khi bấm giờ (không tính vào kết quả)
BASELINE_SEG_S_KRAKEN7_CPU = 459.5
ANCHORS = ["巫山高", "謝眺", "恭惟某官"]

import subprocess, sys, os, json, pathlib, difflib, datetime

WORK = pathlib.Path("/content/kraken_bench"); WORK.mkdir(parents=True, exist_ok=True)
OUT = pathlib.Path("/content/results"); OUT.mkdir(parents=True, exist_ok=True)
VENV = WORK / "venv311"
PY = VENV / "bin" / "python"
REPO = WORK / "CHAT_models"
CHAT_SHA = "9b86ab6c78dc460585c8f63f7149e77e2863419c"


def sh(cmd, timeout=None):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError(f"lệnh lỗi (exit {r.returncode})")
    return r


sh(f"{sys.executable} -m pip install -q uv")
if not PY.exists():
    sh(f"{sys.executable} -m uv venv --python 3.11 {VENV}")
sh(f"{sys.executable} -m uv pip install --python {PY} -q 'kraken==4.3.13' 'torch==2.0.1' 'torchvision==0.15.2' 'numpy<2'")
freeze = sh(f"{sys.executable} -m uv pip freeze --python {PY}").stdout.splitlines()

if not REPO.exists():
    sh(f"git clone -q https://github.com/colibrisson/CHAT_models.git {REPO}")
sh(f"git -C {REPO} checkout -q {CHAT_SHA}")
for w in ("chat_seg.mlmodel", "chat_rec.mlmodel"):
    size = (REPO / "models" / w).stat().st_size
    assert size > 1_000_000, f"{w} chỉ {size} byte — có thể là pointer LFS"
images = sorted((REPO / "demo").glob("*.png"))
print("ảnh:", [p.name for p in images])

BENCH_SRC = """
import argparse, json, time, sys, platform
from importlib.metadata import version
import torch
from PIL import Image
from kraken import blla, rpred
from kraken.lib import vgsl, models

ap = argparse.ArgumentParser()
ap.add_argument("--device"); ap.add_argument("--threads", type=int); ap.add_argument("--warmup", type=int, default=0)
ap.add_argument("--seg"); ap.add_argument("--rec"); ap.add_argument("--out"); ap.add_argument("images", nargs="+")
a = ap.parse_args()
torch.set_num_threads(a.threads)
if a.device.startswith("cuda") and not torch.cuda.is_available():
    sys.exit("CUDA được yêu cầu nhưng torch trong venv không thấy GPU")

def sync():
    if a.device.startswith("cuda"):
        torch.cuda.synchronize()

def check_line_direction(seg):
    for line in seg["lines"]:
        if line["baseline"][0][1] > line["baseline"][-1][1]:
            line["baseline"].reverse()
    return seg

def binarize(path):
    img = Image.open(path).convert("L")
    return img.point(lambda x: 0 if x < 128 else 255, "1")

t = time.perf_counter()
seg_model = vgsl.TorchVGSLModel.load_model(a.seg)
rec_model = models.load_any(a.rec, device=a.device)
sync(); load_s = time.perf_counter() - t

for _ in range(a.warmup):
    blla.segment(binarize(a.images[0]), text_direction="vertical-rl", model=seg_model, device=a.device); sync()

per_image = []
for path in a.images:
    img = binarize(path)
    sync(); t = time.perf_counter()
    seg = blla.segment(img, text_direction="vertical-rl", model=seg_model, device=a.device)
    sync(); seg_s = time.perf_counter() - t
    seg = check_line_direction(seg)
    t = time.perf_counter()
    lines = [getattr(r, "prediction", str(r)) for r in rpred.rpred(rec_model, img, seg)]
    sync(); rec_s = time.perf_counter() - t
    per_image.append({"image": path.rsplit("/", 1)[-1], "size": list(img.size), "n_lines": len(seg["lines"]),
                      "seg_s": seg_s, "rec_s": rec_s, "total_s": seg_s + rec_s, "lines": lines})

json.dump({"device": a.device, "threads": a.threads, "warmup": a.warmup, "load_s": load_s,
           "env": {"python": platform.python_version(), "kraken": version("kraken"), "torch": torch.__version__,
                   "cuda": torch.version.cuda,
                   "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                   "cpu_count": __import__("os").cpu_count()},
           "per_image": per_image}, open(a.out, "w"), ensure_ascii=False, indent=1)
"""
(WORK / "bench.py").write_text(BENCH_SRC)


def run(device, threads, imgs, warmup, timeout=None):
    out = OUT / f"C_kraken_{device.replace(':', '')}.json"
    cmd = (f"cd {REPO} && {PY} {WORK / 'bench.py'} --device {device} --threads {threads} --warmup {warmup} "
           f"--seg models/chat_seg.mlmodel --rec models/chat_rec.mlmodel --out {out} " + " ".join(str(p) for p in imgs))
    try:
        sh(cmd, timeout=timeout)
    except subprocess.TimeoutExpired:
        print(f"⏱ {device}: quá {timeout} s — ghi nhận là timeout")
        return {"device": device, "timeout_s": timeout}
    return json.load(open(out))


runs = {}
if RUN_GPU:
    runs["gpu"] = run("cuda:0", os.cpu_count(), images, GPU_WARMUP)
if RUN_CPU:
    runs["cpu"] = run("cpu", CPU_THREADS, images[:CPU_MAX_IMAGES], 0, timeout=CPU_TIMEOUT_S)

print("\n### Thời gian theo ảnh (giây)\n")
print("| thiết bị | ảnh | số dòng | segmentation | recognition | tổng |")
print("|---|---|---|---|---|---|")
for name, r in runs.items():
    for im in r.get("per_image", []):
        dev_label = r["env"]["gpu"] or f"{r['threads']} thread"
        print(f"| {name} ({dev_label}) | {im['image']} | {im['n_lines']} | "
              f"{im['seg_s']:.1f} | {im['rec_s']:.1f} | {im['total_s']:.1f} |")
    if "timeout_s" in r:
        print(f"| {name} | — | — | timeout > {r['timeout_s']} | — | — |")

comparison = {}
if "per_image" in runs.get("gpu", {}) and "per_image" in runs.get("cpu", {}):
    for c in runs["cpu"]["per_image"]:
        g = next(x for x in runs["gpu"]["per_image"] if x["image"] == c["image"])
        sim = difflib.SequenceMatcher(None, "\n".join(c["lines"]), "\n".join(g["lines"])).ratio()
        comparison[c["image"]] = {"seg_speedup": c["seg_s"] / g["seg_s"], "rec_speedup": c["rec_s"] / g["rec_s"],
                                  "text_similarity_cpu_gpu": sim, "same_n_lines": c["n_lines"] == g["n_lines"]}
        print(f"\n{c['image']}: seg nhanh ×{c['seg_s'] / g['seg_s']:.1f}, rec nhanh ×{c['rec_s'] / g['rec_s']:.1f}; "
              f"độ giống text CPU/GPU = {sim:.3f}; cùng số dòng = {c['n_lines'] == g['n_lines']}")

anchors = {}
if "per_image" in runs.get("gpu", {}):
    gpu_text = "\n".join(l for im in runs["gpu"]["per_image"] for l in im["lines"])
    anchors = {a: a in gpu_text for a in ANCHORS}
    print("\nĐối chứng dương (chuỗi từ log 27/9 có trong output GPU):", anchors,
          "→ TÁI LẬP H2" if all(anchors.values()) else "→ THIẾU chuỗi mốc — đọc lại output trước khi kết luận")
    gseg = [im["seg_s"] for im in runs["gpu"]["per_image"]]
    print(f"Seg GPU trung bình {sum(gseg) / len(gseg):.1f} s/trang so với mốc {BASELINE_SEG_S_KRAKEN7_CPU} s "
          "(mốc là kraken 7.1.1 + CPU sandbox → khác version và phần cứng, chỉ tham khảo; so sánh hợp lệ là CPU vs GPU ở bảng trên)")

json.dump({"experiment": "C_kraken_chat_cpu_vs_gpu", "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
           "chat_models_sha": CHAT_SHA, "config": {"cpu_threads": CPU_THREADS, "cpu_max_images": CPU_MAX_IMAGES,
                                                  "gpu_warmup": GPU_WARMUP},
           "baseline_note": "459.5 s/trang = kraken 7.1.1, CPU sandbox (25/9), không cùng cấu hình",
           "runs": runs, "comparison": comparison, "anchors": anchors, "venv_freeze": freeze},
          open(OUT / "C_kraken_bench.json", "w"), ensure_ascii=False, indent=1)
print("\nĐã lưu /content/results/C_kraken_bench.json")

In [ ]:
# Cell cuối — nén /content/results để tải về (kèm notebook có output)
import shutil, pathlib
p = pathlib.Path("/content/results")
if p.exists() and any(p.iterdir()):
    z = shutil.make_archive("/content/hannom_model_experiments_results", "zip", p)
    print("Đã tạo", z)
    try:
        from google.colab import files
        files.download(z)
    except ImportError:
        print("Không chạy trên Colab — tự lấy file zip ở trên.")
else:
    print("Chưa có kết quả trong /content/results — chạy Cell A/B/C trước.")